In [1]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import IntegerComparator
from math import ceil, log2

from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

In [2]:
def qft_on(circ: QuantumCircuit, qubits):
    n = len(qubits)
    for j in range(n):
        qj = qubits[j]
        circ.h(qj)
        for k in range(j + 1, n):
            qk = qubits[k]
            angle = np.pi / (2 ** (k - j))
            circ.cp(angle, qk, qj)
    # bit reversal
    for j in range(n // 2):
        circ.swap(qubits[j], qubits[n - 1 - j])

In [3]:
def iqft_on(circ: QuantumCircuit, qubits):
    n = len(qubits)
    # undo bit-reversal first
    for j in range(n // 2):
        circ.swap(qubits[j], qubits[n - 1 - j])

    # inverse rotations
    for j in reversed(range(n)):
        qj = qubits[j]
        for k in reversed(range(j + 1, n)):
            qk = qubits[k]
            angle = -np.pi / (2 ** (k - j))  # negative angle
            circ.cp(angle, qk, qj)
        circ.h(qj)

In [4]:
def add_classical_constant_on(circ: QuantumCircuit, qubits, a: int):
    """
    |x> → |x + a (mod 2^n)>
    qubits: MSB → LSB
    """
    qft_on(circ, qubits)
    for j, qj in enumerate(qubits):
        theta = +2 * np.pi * a / (2 ** (j + 1))
        circ.p(theta, qj)
    iqft_on(circ, qubits)

In [5]:
def sub_classical_constant_on(circ: QuantumCircuit, qubits, a: int):
    """
    |x> → |x - a (mod 2^n)>
    """
    qft_on(circ, qubits)
    for j, qj in enumerate(qubits):
        theta = -2 * np.pi * a / (2 ** (j + 1))
        circ.p(theta, qj)
    iqft_on(circ, qubits)

In [6]:
def init_classical_on(circ: QuantumCircuit, reg, value: int):
    """
    reg[0] = MSB, reg[-1] = LSB
    value를 이 순서에 맞게 올려줌.
    """
    n = len(reg)
    bits = format(value, f"0{n}b")  # MSB ... LSB
    for j, b in enumerate(bits):
        if b == "1":
            circ.x(reg[j])

In [7]:
def controlled_sub_classical_on(circ: QuantumCircuit, control, qubits, a: int):
    """
    control=1 일 때만 |x> -> |x - a (mod 2^n)>
    qubits: MSB → LSB
    """
    # 1) QFT
    qft_on(circ, qubits)

    # 2) control이 1일 때만 phase 적용
    for j, qj in enumerate(qubits):
        theta = -2 * np.pi * a / (2 ** (j + 1))
        circ.cp(theta, control, qj)

    # 3) iQFT
    iqft_on(circ, qubits)

In [8]:
def controlled_add_classical_on(circ: QuantumCircuit, control, qubits, a: int):
    """
    control=1 일 때만 |x> -> |x + a (mod 2^n)>
    qubits: MSB → LSB
    """
    if a == 0:
        return  # 0이면 아무 것도 안 해도 됨

    # 1) QFT
    qft_on(circ, qubits)

    # 2) control이 1일 때만 phase 적용
    for j, qj in enumerate(qubits):
        theta = +2 * np.pi * a / (2 ** (j + 1))
        circ.cp(theta, control, qj)

    # 3) iQFT
    iqft_on(circ, qubits)

In [9]:
def build_qtg_with_profit_phase(weights, profits, capacity,
                                theta=np.pi/2,
                                profit_scale=None):
    """
    QTG + profit을 한 큐비트 phase accumulator로 표현하는 버전.

    레지스터:
      path[0..n-1]   : 아이템 선택 비트
      cap[0..n_cap-1]: 잔여 capacity (정수, MSB..LSB)
      pphase         : profit phase accumulator (1 qubit, |+>에서 시작)
      ge             : comparator flag
      wcmp[...]      : comparator ancilla

    각 아이템 m에 대해:
      1) cap >= w_m ?  → ge 에 기록 (IntegerComparator, cap[::-1] 사용)
      2) ge==1 인 branch에서만 path[m]에 R_y(theta)
      3) comparator uncompute
      4) path[m]==1 인 branch에서 cap -= w_m
      5) path[m]==1 인 branch에서 pphase에 Rz( 2π * p_m / profit_scale )
    """

    assert len(weights) == len(profits)
    n = len(weights)

    # ----- capacity 비트 수 -----
    n_cap = max(1, ceil(log2(capacity + 1)))

    # ----- profit 스케일 -----
    #   모든 아이템을 다 넣었을 때 Σ p_m <= profit_scale 이 되도록.
    #   (wrap-around 피하려고 기본값은 Σp_m)
    if profit_scale is None:
        profit_scale = sum(profits) if sum(profits) > 0 else 1

    # ----- comparator용 ancilla 개수 -----
    dummy_cmp = IntegerComparator(num_state_qubits=n_cap, value=0, geq=True)
    n_anc = len(dummy_cmp.qregs[2]) if len(dummy_cmp.qregs) > 2 else 0

    # ----- 레지스터 정의 -----
    path   = QuantumRegister(n,      "path")
    cap    = QuantumRegister(n_cap,  "cap")
    pphase = QuantumRegister(1,      "pphase")   # profit phase qubit
    ge     = QuantumRegister(1,      "ge")
    if n_anc > 0:
        wcmp = QuantumRegister(n_anc, "wcmp")
        qc = QuantumCircuit(path, cap, pphase, ge, wcmp)
    else:
        wcmp = None
        qc = QuantumCircuit(path, cap, pphase, ge)

    # ----- 초기 상태 -----
    # cap = |capacity>, pphase = |+>, path는 전부 0
    init_classical_on(qc, cap, capacity)
    qc.h(pphase[0])

    # ----- 아이템별 QTG step -----
    for m, (w_m, p_m) in enumerate(zip(weights, profits)):

        # (1) cap >= w_m ?  (LSB-first로 comparator에 넘김)
        cmp_circ = IntegerComparator(num_state_qubits=n_cap,
                                     value=w_m,
                                     geq=True)

        cap_for_cmp = list(cap)[::-1]  # LSB ... MSB
        if n_anc > 0:
            cmp_qubits = cap_for_cmp + [ge[0]] + list(wcmp)
        else:
            cmp_qubits = cap_for_cmp + [ge[0]]

        qc.compose(cmp_circ, cmp_qubits, inplace=True)

        # (2) ge==1 인 branch에서만 path[m]에 R_y(theta)
        qc.cry(theta, ge[0], path[m])

        # (3) comparator uncompute
        qc.compose(cmp_circ.inverse(), cmp_qubits, inplace=True)

        # (4) path[m]==1 이면 cap -= w_m
        controlled_sub_classical_on(qc, path[m], cap, w_m)

        # (5) path[m]==1 이면 pphase에 Rz(angle) 추가 (profit 누적)
        angle = np.pi * (p_m / profit_scale)
        qc.crz(angle, path[m], pphase[0])

    return qc, profit_scale

In [10]:
def print_full_statevector_clean(sv, threshold=1e-6, forward=False):
    """
    범용 Statevector 출력 함수.
    - forward=False  : MSB->LSB (사람 기준, Qiskit 기준)
    - forward=True   : LSB->MSB (QFT/ADD/SUB 디버깅용)
                      + index도 LSB 기준 숫자 증가 순으로 정렬
    
    """
    if not isinstance(sv, Statevector):
        sv = Statevector(sv)
    
    amps = sv.data
    n = sv.num_qubits

    print(f"Full Cleaned Statevector (threshold={threshold}, forward={forward})")
    print("-" * 60)

    # 1) 인덱스 리스트 준비
    indices = list(range(len(amps)))

    if forward:
        # ★ forward=True 일 때는 index를 비트 뒤집은 기준으로 정렬해야 함
        def bit_reverse(i):
            raw = format(i, f'0{n}b')  # q2 q1 q0
            rev = raw[::-1]           # q0 q1 q2
            return int(rev, 2)        # rev를 다시 숫자로 해석

        indices.sort(key=lambda i: bit_reverse(i))
    else:
        # 기본은 기존 인덱스 순서 그대로 출력
        pass

    # 2) 출력
    for i in indices:
        amp = amps[i]
        raw = format(i, f'0{n}b')      # q2 q1 q0

        if forward:
            bitstring = raw[::-1]      # LSB->MSB
        else:
            bitstring = raw            # MSB->LSB

        if abs(amp) < threshold:
            # print(f"|{bitstring}> : 0")
            pass
        else:
            print(f"|{bitstring}> : {amp.real:+.6f}{amp.imag:+.6f}j")

    print("-" * 60)

In [11]:
weights  = [3, 2, 4]
profits  = [6, 2, 5]   # 예시
capacity = 5

qc, P_scale = build_qtg_with_profit_phase(
    weights, profits, capacity,
    theta=np.pi/2,
    profit_scale=None,   # 기본값: sum(profits)=13
)
print(qc.draw())

                     ┌─────────┐                                       »
path_0: ─────────────┤ Ry(π/2) ├───────────────────────────────────────»
                     └────┬────┘                                       »
path_1: ──────────────────┼────────────────────────────────────────────»
                          │                                            »
path_2: ──────────────────┼────────────────────────────────────────────»
        ┌───┐┌──────┐     │     ┌─────────┐┌───┐                       »
 cap_0: ┤ X ├┤2     ├─────┼─────┤2        ├┤ H ├─■────────■────────────»
        └───┘│      │     │     │         │└───┘ │P(π/2)  │       ┌───┐»
 cap_1: ─────┤1     ├─────┼─────┤1        ├──────■────────┼───────┤ H ├»
        ┌───┐│      │     │     │         │               │P(π/4) └───┘»
 cap_2: ┤ X ├┤0     ├─────┼─────┤0        ├───────────────■────────────»
        ├───┤│      │     │     │         │                            »
pphase: ┤ H ├┤  cmp ├─────┼─────┤  cmp_dg ├────────

In [12]:
sv = Statevector.from_instruction(qc)
print_full_statevector_clean(sv, threshold=1e-6, forward=True)
print("profit_scale =", P_scale)

Full Cleaned Statevector (threshold=1e-06, forward=True)
------------------------------------------------------------
|0001010000> : +0.250000-0.000000j
|0001011000> : +0.250000-0.000000j
|0010010000> : +0.205746-0.142016j
|0010011000> : +0.205746+0.142016j
|0100110000> : +0.343280-0.084611j
|0100111000> : +0.343280+0.084611j
|1000100000> : +0.264639-0.234449j
|1000101000> : +0.264639+0.234449j
|1100000000> : +0.200841-0.290969j
|1100001000> : +0.200841+0.290969j
------------------------------------------------------------
profit_scale = 13


In [13]:
from collections import defaultdict
from qiskit.quantum_info import Statevector
from qiskit import QuantumCircuit

def path_phase_probs_Xbasis_from_sv(sv: Statevector,
                                    n_path: int,
                                    n_cap: int):
    """
    statevector에서
      - path 비트 (0..n_path-1)
      - profit phase qubit (pphase)
    에 대해, X-basis(|+>,|->) 로 측정했을 때의
    joint prob P(path=x, phase=+/-) 를 계산.

    레지스터 순서: QuantumCircuit(path, cap, pphase, ge, wcmp...)
    """
    n_total = sv.num_qubits
    anc_index = n_path + n_cap  # pphase qubit index

    # 1) ancilla에 H를 적용해서 X basis로 회전
    rot = QuantumCircuit(n_total)
    rot.h(anc_index)
    sv_rot = sv.evolve(rot)   # Rz 로 들어간 profit phase가 |+>,|->확률로 바뀜

    amps = sv_rot.data
    probs = defaultdict(lambda: [0.0, 0.0])  # [P(phase=0), P(phase=1)]

    for idx, amp in enumerate(amps):
        p = abs(amp)**2
        if p < 1e-15:
            continue

        bits = format(idx, f"0{n_total}b")[::-1]  # q0..q_{n-1}
        path_bits = bits[:n_path]                 # path[0..]
        anc_bit = bits[anc_index]                 # pphase (X-basis에서 0=|+>,1=|->)

        key = "".join(path_bits)
        probs[key][int(anc_bit)] += p

    return probs

In [14]:
# QTG + profit phase까지 포함된 상태벡터
sv = Statevector.from_instruction(qc)

# path별, X-basis에서 phase (+/−) joint prob 계산
n_path = len(weights)
n_cap  = max(1, ceil(log2(capacity + 1)))

probs = path_phase_probs_Xbasis_from_sv(sv, n_path=n_path, n_cap=n_cap)

for path in sorted(probs.keys()):
    p_plus, p_minus = probs[path]        # joint: P(path, phase=+/-)
    p_path = p_plus + p_minus            # P(path)
    if p_path > 1e-15:
        # 조건부 확률 P(phase | path)
        cond_plus  = p_plus  / p_path
        cond_minus = p_minus / p_path
    else:
        cond_plus = cond_minus = 0.0

    print(f"path {path}:")
    print(f"  joint : P(+,x)={p_plus:.6f}, P(-,x)={p_minus:.6f}, P(x)={p_path:.6f}")
    print(f"  given : P(+|x)={cond_plus:.6f}, P(-|x)={cond_minus:.6f}")

path 000:
  joint : P(+,x)=0.125000, P(-,x)=0.000000, P(x)=0.125000
  given : P(+|x)=1.000000, P(-|x)=0.000000
path 001:
  joint : P(+,x)=0.084663, P(-,x)=0.040337, P(x)=0.125000
  given : P(+|x)=0.677302, P(-|x)=0.322698
path 010:
  joint : P(+,x)=0.235682, P(-,x)=0.014318, P(x)=0.250000
  given : P(+|x)=0.942728, P(-|x)=0.057272
path 100:
  joint : P(+,x)=0.140067, P(-,x)=0.109933, P(x)=0.250000
  given : P(+|x)=0.560268, P(-|x)=0.439732
path 110:
  joint : P(+,x)=0.080674, P(-,x)=0.169326, P(x)=0.250000
  given : P(+|x)=0.322698, P(-|x)=0.677302
